# Klasifikasi Hate Speech dengan Model Multibahasa

## Persiapan

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import gc
import json
import random
import shutil
import sys
import tempfile
from datetime import datetime
from functools import partial
from pathlib import Path

import emoji
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import transformers
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

## Konfigurasi eksperimen

In [2]:
model_names = [
    "FacebookAI/xlm-roberta-base",
    "google-bert/bert-base-multilingual-uncased",
    "cardiffnlp/twitter-xlm-roberta-base",
]
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

num_train_epochs = 10
early_stopping_patience = 3
experiment_results = []
experiment_started_at = datetime.now().astimezone().isoformat(timespec="seconds")
checkpoint_dirs = {
    name: Path(tempfile.mkdtemp(prefix="stereoqueereval_checkpoint_"))
    for name in model_names
}
artifact_dirs = {
    name: Path(tempfile.mkdtemp(prefix="stereoqueereval_artifacts_"))
    for name in model_names
}


## Data

In [3]:
project_dir = Path(".") if Path("data").exists() else Path("..")
input_dir = project_dir / "data"

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU tidak terdeteksi. Training dihentikan.")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

train_df = pd.read_excel(input_dir / "train.xlsx")[["StereoQueerEval_id", "yt_comment", "hate_speech"]]
val_df = pd.read_excel(input_dir / "val.xlsx")[["StereoQueerEval_id", "yt_comment", "hate_speech"]]
test_df = pd.read_excel(input_dir / "test.xlsx")[["StereoQueerEval_id", "yt_comment", "hate_speech"]]

label2id = {"no": 0, "yes_implicit": 1, "yes_explicit": 2}
id2label = {value: key for key, value in label2id.items()}
emoji_pattern = emoji.get_emoji_regexp()

for data in [train_df, val_df, test_df]:
    data["text"] = data["yt_comment"].fillna("").astype(str).apply(
        lambda text: emoji_pattern.sub(" [EMOJI] ", text).casefold()
    )
    data["labels"] = data["hate_speech"].map(label2id)
    if data["labels"].isna().any():
        raise ValueError("Ditemukan label hate_speech yang tidak cocok dengan label2id.")
    data["labels"] = data["labels"].astype(int)

print("Jumlah data:", {"train": len(train_df), "validation": len(val_df), "test": len(test_df)})

GPU: NVIDIA GeForce RTX 3060 Ti
CUDA: 13.0
Jumlah data: {'train': 5343, 'validation': 763, 'test': 1521}


## Metrik evaluasi dan fungsi pembersihan memori

In [4]:
def compute_metrics(eval_pred):
    y_true = eval_pred.label_ids
    y_pred = np.argmax(eval_pred.predictions, axis=1)
    return {"macro_f1": f1_score(y_true, y_pred, average="macro")}


def cleanup():
    """Bersihkan objek tidak terpakai di RAM dan cache GPU."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## XLM-RoBERTa-base

In [5]:
model_name = model_names[0]
checkpoint_dir = checkpoint_dirs[model_name]
artifact_dir = artifact_dirs[model_name]


### Tokenisasi dan class weights

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_data = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
val_data = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False)
test_data = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)
tokenize = lambda batch: tokenizer(batch["text"], truncation=True, max_length=384)
train_data = train_data.map(tokenize, batched=True, remove_columns=["text"], desc="Tokenisasi train")
val_data = val_data.map(tokenize, batched=True, remove_columns=["text"], desc="Tokenisasi validation")
test_data = test_data.map(tokenize, batched=True, remove_columns=["text"], desc="Tokenisasi test")

class_counts = np.bincount(train_df["labels"], minlength=len(label2id))
class_weights = torch.tensor(
    len(train_df) / (len(label2id) * class_counts), dtype=torch.float32
)
print("Class weights:", dict(zip(label2id, class_weights.tolist())))



Tokenisasi train:   0%|          | 0/5343 [00:00<?, ? examples/s]

Tokenisasi validation:   0%|          | 0/763 [00:00<?, ? examples/s]

Tokenisasi test:   0%|          | 0/1521 [00:00<?, ? examples/s]

Class weights: {'no': 0.6884422302246094, 'yes_implicit': 1.3816912174224854, 'yes_explicit': 1.21404230594635}


### Fine-tuning

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
)
# Simpan nama parameter runtime agar checkpoint cocok saat dimuat ulang.
model.save_pretrained = partial(
    model.save_pretrained, save_original_format=False
)

training_args = TrainingArguments(
    output_dir=str(checkpoint_dir / "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,
    weight_decay=0.01,
    seed=seed,
    data_seed=seed,
    full_determinism=True,
    num_train_epochs=num_train_epochs,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    fp16=True,
    disable_tqdm=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)],
    compute_loss_func=lambda outputs, labels, num_items_in_batch=None: torch.nn.functional.cross_entropy(
        outputs.logits.float(), labels, weight=class_weights.to(outputs.logits.device)
    ),
)
print()
print("Model:", model_name)
print("Training device:", trainer.args.device)
if trainer.args.device.type != "cuda":
    raise RuntimeError("Trainer tidak menggunakan CUDA.")
trainer.train()



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model: FacebookAI/xlm-roberta-base
Training device: cuda:0


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,1.000842,0.487887
2,1.042769,0.965275,0.513812
3,0.912254,0.966601,0.548358
4,0.912254,1.067738,0.525543
5,0.752839,1.044023,0.550156
6,0.649884,1.115070,0.551025
7,0.649884,1.253087,0.559666
8,0.530251,1.347714,0.536579
9,0.458500,1.406126,0.556115
10,0.458500,1.475419,0.538307


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3340, training_loss=0.6925063538694096, metrics={'train_runtime': 573.2647, 'train_samples_per_second': 93.203, 'train_steps_per_second': 5.826, 'total_flos': 4211606629912878.0, 'train_loss': 0.6925063538694096, 'epoch': 10.0})

### Evaluasi validation dan test

In [8]:
val_predictions = trainer.predict(val_data)
y_val = val_predictions.label_ids
y_val_pred = val_predictions.predictions.argmax(axis=1)
test_predictions = trainer.predict(test_data)
y_test = test_predictions.label_ids
y_test_pred = test_predictions.predictions.argmax(axis=1)
test_probabilities = torch.softmax(torch.tensor(test_predictions.predictions), dim=1).numpy()

metrics = {}
for split_name, y_true, y_pred in [
    ("validation", y_val, y_val_pred),
    ("test", y_test, y_test_pred),
]:
    metrics[split_name] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "classification_report": classification_report(
            y_true, y_pred, labels=list(id2label), target_names=list(label2id),
            output_dict=True, zero_division=0
        ),
    }
    print(f"{model_name} | {split_name.title()} accuracy: {metrics[split_name]['accuracy']:.4f}")
    print(f"{model_name} | {split_name.title()} macro-F1: {metrics[split_name]['macro_f1']:.4f}")
    print(classification_report(
        y_true, y_pred, labels=list(id2label), target_names=list(label2id), zero_division=0
    ))



FacebookAI/xlm-roberta-base | Validation accuracy: 0.6160
FacebookAI/xlm-roberta-base | Validation macro-F1: 0.5597
              precision    recall  f1-score   support

          no       0.70      0.77      0.73       369
yes_implicit       0.45      0.29      0.36       184
yes_explicit       0.56      0.63      0.59       210

    accuracy                           0.62       763
   macro avg       0.57      0.56      0.56       763
weighted avg       0.60      0.62      0.60       763

FacebookAI/xlm-roberta-base | Test accuracy: 0.6134
FacebookAI/xlm-roberta-base | Test macro-F1: 0.5531
              precision    recall  f1-score   support

          no       0.69      0.78      0.73       738
yes_implicit       0.47      0.27      0.34       367
yes_explicit       0.55      0.63      0.59       416

    accuracy                           0.61      1521
   macro avg       0.57      0.56      0.55      1521
weighted avg       0.60      0.61      0.60      1521



### Simpan artefak ZIP

In [9]:
(artifact_dir / "evaluation_results.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
)
pd.DataFrame([
    {"split": split_name, **{key: value for key, value in result.items() if key != "classification_report"}}
    for split_name, result in metrics.items()
]).to_csv(artifact_dir / "evaluation_summary.csv", index=False)

training_log = pd.DataFrame(trainer.state.log_history)
training_log.to_csv(artifact_dir / "training_log.csv", index=False)
(artifact_dir / "training_log.json").write_text(
    training_log.to_json(orient="records", indent=2), encoding="utf-8"
)

train_loss = [row for row in trainer.state.log_history if "loss" in row and "eval_loss" not in row]
eval_log = [row for row in trainer.state.log_history if "eval_loss" in row]
plt.figure(figsize=(8, 5))
if train_loss:
    plt.plot([row["step"] for row in train_loss], [row["loss"] for row in train_loss], label="Train loss")
if eval_log:
    plt.plot([row["step"] for row in eval_log], [row["eval_loss"] for row in eval_log], marker="o", label="Validation loss")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title(f"Training and validation loss - {model_name}")
plt.legend()
plt.tight_layout()
plt.savefig(artifact_dir / "training_loss.png", dpi=150)
plt.close()

macro_logs = [row for row in eval_log if "eval_macro_f1" in row]
plt.figure(figsize=(8, 5))
if macro_logs:
    plt.plot([row["epoch"] for row in macro_logs], [row["eval_macro_f1"] for row in macro_logs], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title(f"Validation macro-F1 per epoch - {model_name}")
plt.tight_layout()
plt.savefig(artifact_dir / "validation_macro_f1.png", dpi=150)
plt.close()

for split_name, y_true, y_pred in [
    ("validation", y_val, y_val_pred),
    ("test", y_test, y_test_pred),
]:
    matrix = confusion_matrix(y_true, y_pred, labels=list(id2label))
    pd.DataFrame(matrix, index=list(label2id), columns=list(label2id)).to_csv(
        artifact_dir / f"confusion_matrix_{split_name}.csv"
    )
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        matrix, annot=True, fmt="d", cmap="Blues",
        xticklabels=list(label2id), yticklabels=list(label2id)
    )
    plt.xlabel("Prediksi")
    plt.ylabel("Label aktual")
    plt.title(f"Confusion matrix - {split_name.title()}")
    plt.tight_layout()
    plt.savefig(artifact_dir / f"confusion_matrix_{split_name}.png", dpi=150)
    plt.close()

test_prediction = pd.DataFrame({
    "StereoQueerEval_id": test_df["StereoQueerEval_id"],
    "yt_comment": test_df["yt_comment"],
    "true_label": [id2label[int(label)] for label in y_test],
    "predicted_label": [id2label[int(label)] for label in y_test_pred],
})
for class_id, class_name in id2label.items():
    test_prediction[f"prob_{class_name}"] = test_probabilities[:, class_id]
test_prediction.to_csv(artifact_dir / "test_prediction.csv", index=False, encoding="utf-8-sig")

manifest = {
    "experiment_name": f"{model_name} hate speech classification with comment preprocessing and class weights",
    "started_at": experiment_started_at,
    "completed_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "model": model_name,
    "task": "three-class hate speech classification",
    "input_feature": "yt_comment only",
    "preprocessing": ["replace emoji with [EMOJI]", "Unicode casefold"],
    "label2id": label2id,
    "class_counts_train": train_df["hate_speech"].value_counts().to_dict(),
    "class_weights_train": dict(zip(label2id, class_weights.tolist())),
    "dataset_sizes": {"train": len(train_df), "validation": len(val_df), "test": len(test_df)},
    "parameters": {
        "learning_rate": trainer.args.learning_rate,
        "weight_decay": trainer.args.weight_decay,
        "num_train_epochs": trainer.args.num_train_epochs,
        "early_stopping_patience": early_stopping_patience,
        "per_device_train_batch_size": trainer.args.per_device_train_batch_size,
        "per_device_eval_batch_size": trainer.args.per_device_eval_batch_size,
        "max_length": 384,
        "fp16": trainer.args.fp16,
        "seed": seed,
        "data_seed": seed,
        "full_determinism": trainer.args.full_determinism,
        "metric_for_best_model": trainer.args.metric_for_best_model,
    },
    "best_validation_macro_f1": trainer.state.best_metric,
    "evaluation_summary": {
        split: {key: value for key, value in result.items() if key != "classification_report"}
        for split, result in metrics.items()
    },
    "environment": {
        "python": sys.version,
        "transformers": transformers.__version__,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "emoji": emoji.__version__,
    },
}
(artifact_dir / "experiment_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)

results_dir = project_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)
model_slug = model_name.rsplit("/", 1)[-1]
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = results_dir / f"{model_slug}_{timestamp}.zip"
shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=artifact_dir)
print("Artifact ZIP:", zip_path.resolve())
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024 * 1024), 2))

experiment_results.append({
    "model": model_name,
    "validation_macro_f1": metrics["validation"]["macro_f1"],
    "test_macro_f1": metrics["test"]["macro_f1"],
    "artifact_zip": str(zip_path.resolve()),
})

del trainer, model, tokenizer, train_data, val_data, test_data, tokenize
shutil.rmtree(artifact_dir, ignore_errors=True)
shutil.rmtree(checkpoint_dir, ignore_errors=True)
cleanup()
print("RAM dan cache GPU dibersihkan setelah:", model_name)


Artifact ZIP: E:\Projek\PTM\TBP\Analisis\results\xlm-roberta-base_20261019_120700.zip
ZIP size (MB): 0.3
RAM dan cache GPU dibersihkan setelah: FacebookAI/xlm-roberta-base


## BERT-base-multilingual-uncased

In [10]:
model_name = model_names[1]
checkpoint_dir = checkpoint_dirs[model_name]
artifact_dir = artifact_dirs[model_name]


### Tokenisasi dan class weights

In [11]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_data = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
val_data = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False)
test_data = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)
tokenize = lambda batch: tokenizer(batch["text"], truncation=True, max_length=384)
train_data = train_data.map(tokenize, batched=True, remove_columns=["text"], desc="Tokenisasi train")
val_data = val_data.map(tokenize, batched=True, remove_columns=["text"], desc="Tokenisasi validation")
test_data = test_data.map(tokenize, batched=True, remove_columns=["text"], desc="Tokenisasi test")

class_counts = np.bincount(train_df["labels"], minlength=len(label2id))
class_weights = torch.tensor(
    len(train_df) / (len(label2id) * class_counts), dtype=torch.float32
)
print("Class weights:", dict(zip(label2id, class_weights.tolist())))



Tokenisasi train:   0%|          | 0/5343 [00:00<?, ? examples/s]

Tokenisasi validation:   0%|          | 0/763 [00:00<?, ? examples/s]

Tokenisasi test:   0%|          | 0/1521 [00:00<?, ? examples/s]

Class weights: {'no': 0.6884422302246094, 'yes_implicit': 1.3816912174224854, 'yes_explicit': 1.21404230594635}


### Fine-tuning

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
)
# Simpan nama parameter runtime agar checkpoint cocok saat dimuat ulang.
model.save_pretrained = partial(
    model.save_pretrained, save_original_format=False
)

training_args = TrainingArguments(
    output_dir=str(checkpoint_dir / "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,
    weight_decay=0.01,
    seed=seed,
    data_seed=seed,
    full_determinism=True,
    num_train_epochs=num_train_epochs,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    fp16=True,
    disable_tqdm=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)],
    compute_loss_func=lambda outputs, labels, num_items_in_batch=None: torch.nn.functional.cross_entropy(
        outputs.logits.float(), labels, weight=class_weights.to(outputs.logits.device)
    ),
)
print()
print("Model:", model_name)
print("Training device:", trainer.args.device)
if trainer.args.device.type != "cuda":
    raise RuntimeError("Trainer tidak menggunakan CUDA.")
trainer.train()



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-multilingual-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model: google-bert/bert-base-multilingual-uncased
Training device: cuda:0


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,1.021340,0.476394
2,1.038294,0.945173,0.524963
3,0.864774,0.949512,0.540458
4,0.864774,1.097964,0.544981
5,0.637484,1.131179,0.543902
6,0.474490,1.276294,0.523749
7,0.474490,1.369101,0.557556
8,0.323964,1.544205,0.535940
9,0.256453,1.577023,0.538125
10,0.256453,1.664352,0.529178


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3340, training_loss=0.558785448816722, metrics={'train_runtime': 456.1424, 'train_samples_per_second': 117.134, 'train_steps_per_second': 7.322, 'total_flos': 4285663293028824.0, 'train_loss': 0.558785448816722, 'epoch': 10.0})

### Evaluasi validation dan test

In [13]:
val_predictions = trainer.predict(val_data)
y_val = val_predictions.label_ids
y_val_pred = val_predictions.predictions.argmax(axis=1)
test_predictions = trainer.predict(test_data)
y_test = test_predictions.label_ids
y_test_pred = test_predictions.predictions.argmax(axis=1)
test_probabilities = torch.softmax(torch.tensor(test_predictions.predictions), dim=1).numpy()

metrics = {}
for split_name, y_true, y_pred in [
    ("validation", y_val, y_val_pred),
    ("test", y_test, y_test_pred),
]:
    metrics[split_name] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "classification_report": classification_report(
            y_true, y_pred, labels=list(id2label), target_names=list(label2id),
            output_dict=True, zero_division=0
        ),
    }
    print(f"{model_name} | {split_name.title()} accuracy: {metrics[split_name]['accuracy']:.4f}")
    print(f"{model_name} | {split_name.title()} macro-F1: {metrics[split_name]['macro_f1']:.4f}")
    print(classification_report(
        y_true, y_pred, labels=list(id2label), target_names=list(label2id), zero_division=0
    ))



google-bert/bert-base-multilingual-uncased | Validation accuracy: 0.5845
google-bert/bert-base-multilingual-uncased | Validation macro-F1: 0.5576
              precision    recall  f1-score   support

          no       0.74      0.66      0.70       369
yes_implicit       0.37      0.45      0.41       184
yes_explicit       0.57      0.57      0.57       210

    accuracy                           0.58       763
   macro avg       0.56      0.56      0.56       763
weighted avg       0.60      0.58      0.59       763

google-bert/bert-base-multilingual-uncased | Test accuracy: 0.5674
google-bert/bert-base-multilingual-uncased | Test macro-F1: 0.5442
              precision    recall  f1-score   support

          no       0.72      0.62      0.67       738
yes_implicit       0.38      0.44      0.41       367
yes_explicit       0.53      0.59      0.56       416

    accuracy                           0.57      1521
   macro avg       0.54      0.55      0.54      1521
weighted avg 

### Simpan artefak ZIP

In [14]:
(artifact_dir / "evaluation_results.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
)
pd.DataFrame([
    {"split": split_name, **{key: value for key, value in result.items() if key != "classification_report"}}
    for split_name, result in metrics.items()
]).to_csv(artifact_dir / "evaluation_summary.csv", index=False)

training_log = pd.DataFrame(trainer.state.log_history)
training_log.to_csv(artifact_dir / "training_log.csv", index=False)
(artifact_dir / "training_log.json").write_text(
    training_log.to_json(orient="records", indent=2), encoding="utf-8"
)

train_loss = [row for row in trainer.state.log_history if "loss" in row and "eval_loss" not in row]
eval_log = [row for row in trainer.state.log_history if "eval_loss" in row]
plt.figure(figsize=(8, 5))
if train_loss:
    plt.plot([row["step"] for row in train_loss], [row["loss"] for row in train_loss], label="Train loss")
if eval_log:
    plt.plot([row["step"] for row in eval_log], [row["eval_loss"] for row in eval_log], marker="o", label="Validation loss")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title(f"Training and validation loss - {model_name}")
plt.legend()
plt.tight_layout()
plt.savefig(artifact_dir / "training_loss.png", dpi=150)
plt.close()

macro_logs = [row for row in eval_log if "eval_macro_f1" in row]
plt.figure(figsize=(8, 5))
if macro_logs:
    plt.plot([row["epoch"] for row in macro_logs], [row["eval_macro_f1"] for row in macro_logs], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title(f"Validation macro-F1 per epoch - {model_name}")
plt.tight_layout()
plt.savefig(artifact_dir / "validation_macro_f1.png", dpi=150)
plt.close()

for split_name, y_true, y_pred in [
    ("validation", y_val, y_val_pred),
    ("test", y_test, y_test_pred),
]:
    matrix = confusion_matrix(y_true, y_pred, labels=list(id2label))
    pd.DataFrame(matrix, index=list(label2id), columns=list(label2id)).to_csv(
        artifact_dir / f"confusion_matrix_{split_name}.csv"
    )
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        matrix, annot=True, fmt="d", cmap="Blues",
        xticklabels=list(label2id), yticklabels=list(label2id)
    )
    plt.xlabel("Prediksi")
    plt.ylabel("Label aktual")
    plt.title(f"Confusion matrix - {split_name.title()}")
    plt.tight_layout()
    plt.savefig(artifact_dir / f"confusion_matrix_{split_name}.png", dpi=150)
    plt.close()

test_prediction = pd.DataFrame({
    "StereoQueerEval_id": test_df["StereoQueerEval_id"],
    "yt_comment": test_df["yt_comment"],
    "true_label": [id2label[int(label)] for label in y_test],
    "predicted_label": [id2label[int(label)] for label in y_test_pred],
})
for class_id, class_name in id2label.items():
    test_prediction[f"prob_{class_name}"] = test_probabilities[:, class_id]
test_prediction.to_csv(artifact_dir / "test_prediction.csv", index=False, encoding="utf-8-sig")

manifest = {
    "experiment_name": f"{model_name} hate speech classification with comment preprocessing and class weights",
    "started_at": experiment_started_at,
    "completed_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "model": model_name,
    "task": "three-class hate speech classification",
    "input_feature": "yt_comment only",
    "preprocessing": ["replace emoji with [EMOJI]", "Unicode casefold"],
    "label2id": label2id,
    "class_counts_train": train_df["hate_speech"].value_counts().to_dict(),
    "class_weights_train": dict(zip(label2id, class_weights.tolist())),
    "dataset_sizes": {"train": len(train_df), "validation": len(val_df), "test": len(test_df)},
    "parameters": {
        "learning_rate": trainer.args.learning_rate,
        "weight_decay": trainer.args.weight_decay,
        "num_train_epochs": trainer.args.num_train_epochs,
        "early_stopping_patience": early_stopping_patience,
        "per_device_train_batch_size": trainer.args.per_device_train_batch_size,
        "per_device_eval_batch_size": trainer.args.per_device_eval_batch_size,
        "max_length": 384,
        "fp16": trainer.args.fp16,
        "seed": seed,
        "data_seed": seed,
        "full_determinism": trainer.args.full_determinism,
        "metric_for_best_model": trainer.args.metric_for_best_model,
    },
    "best_validation_macro_f1": trainer.state.best_metric,
    "evaluation_summary": {
        split: {key: value for key, value in result.items() if key != "classification_report"}
        for split, result in metrics.items()
    },
    "environment": {
        "python": sys.version,
        "transformers": transformers.__version__,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "emoji": emoji.__version__,
    },
}
(artifact_dir / "experiment_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)

results_dir = project_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)
model_slug = model_name.rsplit("/", 1)[-1]
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = results_dir / f"{model_slug}_{timestamp}.zip"
shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=artifact_dir)
print("Artifact ZIP:", zip_path.resolve())
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024 * 1024), 2))

experiment_results.append({
    "model": model_name,
    "validation_macro_f1": metrics["validation"]["macro_f1"],
    "test_macro_f1": metrics["test"]["macro_f1"],
    "artifact_zip": str(zip_path.resolve()),
})

del trainer, model, tokenizer, train_data, val_data, test_data, tokenize
shutil.rmtree(artifact_dir, ignore_errors=True)
shutil.rmtree(checkpoint_dir, ignore_errors=True)
cleanup()
print("RAM dan cache GPU dibersihkan setelah:", model_name)


Artifact ZIP: E:\Projek\PTM\TBP\Analisis\results\bert-base-multilingual-uncased_20261019_121445.zip
ZIP size (MB): 0.3
RAM dan cache GPU dibersihkan setelah: google-bert/bert-base-multilingual-uncased


## Twitter-XLM-RoBERTa-base

In [15]:
model_name = model_names[2]
checkpoint_dir = checkpoint_dirs[model_name]
artifact_dir = artifact_dirs[model_name]


### Tokenisasi dan class weights

In [16]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_data = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
val_data = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False)
test_data = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)
tokenize = lambda batch: tokenizer(batch["text"], truncation=True, max_length=384)
train_data = train_data.map(tokenize, batched=True, remove_columns=["text"], desc="Tokenisasi train")
val_data = val_data.map(tokenize, batched=True, remove_columns=["text"], desc="Tokenisasi validation")
test_data = test_data.map(tokenize, batched=True, remove_columns=["text"], desc="Tokenisasi test")

class_counts = np.bincount(train_df["labels"], minlength=len(label2id))
class_weights = torch.tensor(
    len(train_df) / (len(label2id) * class_counts), dtype=torch.float32
)
print("Class weights:", dict(zip(label2id, class_weights.tolist())))



Tokenisasi train:   0%|          | 0/5343 [00:00<?, ? examples/s]

Tokenisasi validation:   0%|          | 0/763 [00:00<?, ? examples/s]

Tokenisasi test:   0%|          | 0/1521 [00:00<?, ? examples/s]

Class weights: {'no': 0.6884422302246094, 'yes_implicit': 1.3816912174224854, 'yes_explicit': 1.21404230594635}


### Fine-tuning

In [17]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
)
# Simpan nama parameter runtime agar checkpoint cocok saat dimuat ulang.
model.save_pretrained = partial(
    model.save_pretrained, save_original_format=False
)

training_args = TrainingArguments(
    output_dir=str(checkpoint_dir / "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,
    weight_decay=0.01,
    seed=seed,
    data_seed=seed,
    full_determinism=True,
    num_train_epochs=num_train_epochs,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    fp16=True,
    disable_tqdm=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=early_stopping_patience)],
    compute_loss_func=lambda outputs, labels, num_items_in_batch=None: torch.nn.functional.cross_entropy(
        outputs.logits.float(), labels, weight=class_weights.to(outputs.logits.device)
    ),
)
print()
print("Model:", model_name)
print("Training device:", trainer.args.device)
if trainer.args.device.type != "cuda":
    raise RuntimeError("Trainer tidak menggunakan CUDA.")
trainer.train()



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstr


Model: cardiffnlp/twitter-xlm-roberta-base
Training device: cuda:0


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.933741,0.531540
2,0.989766,0.900844,0.584052
3,0.795419,0.969067,0.575995
4,0.795419,1.027345,0.583791
5,0.584442,1.187588,0.563446


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1670, training_loss=0.7620345658170963, metrics={'train_runtime': 274.5173, 'train_samples_per_second': 194.633, 'train_steps_per_second': 12.167, 'total_flos': 2121392528010126.0, 'train_loss': 0.7620345658170963, 'epoch': 5.0})

### Evaluasi validation dan test

In [18]:
val_predictions = trainer.predict(val_data)
y_val = val_predictions.label_ids
y_val_pred = val_predictions.predictions.argmax(axis=1)
test_predictions = trainer.predict(test_data)
y_test = test_predictions.label_ids
y_test_pred = test_predictions.predictions.argmax(axis=1)
test_probabilities = torch.softmax(torch.tensor(test_predictions.predictions), dim=1).numpy()

metrics = {}
for split_name, y_true, y_pred in [
    ("validation", y_val, y_val_pred),
    ("test", y_test, y_test_pred),
]:
    metrics[split_name] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "classification_report": classification_report(
            y_true, y_pred, labels=list(id2label), target_names=list(label2id),
            output_dict=True, zero_division=0
        ),
    }
    print(f"{model_name} | {split_name.title()} accuracy: {metrics[split_name]['accuracy']:.4f}")
    print(f"{model_name} | {split_name.title()} macro-F1: {metrics[split_name]['macro_f1']:.4f}")
    print(classification_report(
        y_true, y_pred, labels=list(id2label), target_names=list(label2id), zero_division=0
    ))



cardiffnlp/twitter-xlm-roberta-base | Validation accuracy: 0.6134
cardiffnlp/twitter-xlm-roberta-base | Validation macro-F1: 0.5841
              precision    recall  f1-score   support

          no       0.75      0.70      0.72       369
yes_implicit       0.42      0.47      0.45       184
yes_explicit       0.58      0.59      0.58       210

    accuracy                           0.61       763
   macro avg       0.58      0.59      0.58       763
weighted avg       0.62      0.61      0.62       763

cardiffnlp/twitter-xlm-roberta-base | Test accuracy: 0.5858
cardiffnlp/twitter-xlm-roberta-base | Test macro-F1: 0.5547
              precision    recall  f1-score   support

          no       0.71      0.67      0.69       738
yes_implicit       0.39      0.41      0.40       367
yes_explicit       0.56      0.59      0.57       416

    accuracy                           0.59      1521
   macro avg       0.55      0.56      0.55      1521
weighted avg       0.59      0.59      0.

### Simpan artefak ZIP

In [19]:
(artifact_dir / "evaluation_results.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
)
pd.DataFrame([
    {"split": split_name, **{key: value for key, value in result.items() if key != "classification_report"}}
    for split_name, result in metrics.items()
]).to_csv(artifact_dir / "evaluation_summary.csv", index=False)

training_log = pd.DataFrame(trainer.state.log_history)
training_log.to_csv(artifact_dir / "training_log.csv", index=False)
(artifact_dir / "training_log.json").write_text(
    training_log.to_json(orient="records", indent=2), encoding="utf-8"
)

train_loss = [row for row in trainer.state.log_history if "loss" in row and "eval_loss" not in row]
eval_log = [row for row in trainer.state.log_history if "eval_loss" in row]
plt.figure(figsize=(8, 5))
if train_loss:
    plt.plot([row["step"] for row in train_loss], [row["loss"] for row in train_loss], label="Train loss")
if eval_log:
    plt.plot([row["step"] for row in eval_log], [row["eval_loss"] for row in eval_log], marker="o", label="Validation loss")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title(f"Training and validation loss - {model_name}")
plt.legend()
plt.tight_layout()
plt.savefig(artifact_dir / "training_loss.png", dpi=150)
plt.close()

macro_logs = [row for row in eval_log if "eval_macro_f1" in row]
plt.figure(figsize=(8, 5))
if macro_logs:
    plt.plot([row["epoch"] for row in macro_logs], [row["eval_macro_f1"] for row in macro_logs], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title(f"Validation macro-F1 per epoch - {model_name}")
plt.tight_layout()
plt.savefig(artifact_dir / "validation_macro_f1.png", dpi=150)
plt.close()

for split_name, y_true, y_pred in [
    ("validation", y_val, y_val_pred),
    ("test", y_test, y_test_pred),
]:
    matrix = confusion_matrix(y_true, y_pred, labels=list(id2label))
    pd.DataFrame(matrix, index=list(label2id), columns=list(label2id)).to_csv(
        artifact_dir / f"confusion_matrix_{split_name}.csv"
    )
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        matrix, annot=True, fmt="d", cmap="Blues",
        xticklabels=list(label2id), yticklabels=list(label2id)
    )
    plt.xlabel("Prediksi")
    plt.ylabel("Label aktual")
    plt.title(f"Confusion matrix - {split_name.title()}")
    plt.tight_layout()
    plt.savefig(artifact_dir / f"confusion_matrix_{split_name}.png", dpi=150)
    plt.close()

test_prediction = pd.DataFrame({
    "StereoQueerEval_id": test_df["StereoQueerEval_id"],
    "yt_comment": test_df["yt_comment"],
    "true_label": [id2label[int(label)] for label in y_test],
    "predicted_label": [id2label[int(label)] for label in y_test_pred],
})
for class_id, class_name in id2label.items():
    test_prediction[f"prob_{class_name}"] = test_probabilities[:, class_id]
test_prediction.to_csv(artifact_dir / "test_prediction.csv", index=False, encoding="utf-8-sig")

manifest = {
    "experiment_name": f"{model_name} hate speech classification with comment preprocessing and class weights",
    "started_at": experiment_started_at,
    "completed_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "model": model_name,
    "task": "three-class hate speech classification",
    "input_feature": "yt_comment only",
    "preprocessing": ["replace emoji with [EMOJI]", "Unicode casefold"],
    "label2id": label2id,
    "class_counts_train": train_df["hate_speech"].value_counts().to_dict(),
    "class_weights_train": dict(zip(label2id, class_weights.tolist())),
    "dataset_sizes": {"train": len(train_df), "validation": len(val_df), "test": len(test_df)},
    "parameters": {
        "learning_rate": trainer.args.learning_rate,
        "weight_decay": trainer.args.weight_decay,
        "num_train_epochs": trainer.args.num_train_epochs,
        "early_stopping_patience": early_stopping_patience,
        "per_device_train_batch_size": trainer.args.per_device_train_batch_size,
        "per_device_eval_batch_size": trainer.args.per_device_eval_batch_size,
        "max_length": 384,
        "fp16": trainer.args.fp16,
        "seed": seed,
        "data_seed": seed,
        "full_determinism": trainer.args.full_determinism,
        "metric_for_best_model": trainer.args.metric_for_best_model,
    },
    "best_validation_macro_f1": trainer.state.best_metric,
    "evaluation_summary": {
        split: {key: value for key, value in result.items() if key != "classification_report"}
        for split, result in metrics.items()
    },
    "environment": {
        "python": sys.version,
        "transformers": transformers.__version__,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "emoji": emoji.__version__,
    },
}
(artifact_dir / "experiment_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)

results_dir = project_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)
model_slug = model_name.rsplit("/", 1)[-1]
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = results_dir / f"{model_slug}_{timestamp}.zip"
shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=artifact_dir)
print("Artifact ZIP:", zip_path.resolve())
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024 * 1024), 2))

experiment_results.append({
    "model": model_name,
    "validation_macro_f1": metrics["validation"]["macro_f1"],
    "test_macro_f1": metrics["test"]["macro_f1"],
    "artifact_zip": str(zip_path.resolve()),
})

del trainer, model, tokenizer, train_data, val_data, test_data, tokenize
shutil.rmtree(artifact_dir, ignore_errors=True)
shutil.rmtree(checkpoint_dir, ignore_errors=True)
cleanup()
print("RAM dan cache GPU dibersihkan setelah:", model_name)


Artifact ZIP: E:\Projek\PTM\TBP\Analisis\results\twitter-xlm-roberta-base_20261019_121932.zip
ZIP size (MB): 0.3
RAM dan cache GPU dibersihkan setelah: cardiffnlp/twitter-xlm-roberta-base


## Ringkasan hasil

In [20]:
print(pd.DataFrame(experiment_results).to_string(index=False))

                                     model  validation_macro_f1  test_macro_f1                                                                          artifact_zip
               FacebookAI/xlm-roberta-base             0.559666       0.553115               E:\Projek\PTM\TBP\Analisis\results\xlm-roberta-base_20261019_120700.zip
google-bert/bert-base-multilingual-uncased             0.557556       0.544244 E:\Projek\PTM\TBP\Analisis\results\bert-base-multilingual-uncased_20261019_121445.zip
       cardiffnlp/twitter-xlm-roberta-base             0.584052       0.554692       E:\Projek\PTM\TBP\Analisis\results\twitter-xlm-roberta-base_20261019_121932.zip
